In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


In [2]:
ravdess_csv = r"D:\SER_Cross\data\processed\ravdess_metadata.csv"
crema_csv   = r"D:\SER_Cross\data\processed\crema_d_metadata.csv"

df_ravdess = pd.read_csv(ravdess_csv)
df_crema   = pd.read_csv(crema_csv)

print(len(df_ravdess), len(df_crema))


1056 6171


In [3]:
# Update RAVDESS paths → standardized audio
df_ravdess["path"] = df_ravdess["path"].apply(
    lambda p: p.replace(
        r"D:\SER_Cross\data\ravdess",
        r"D:\SER_Cross\data\processed\ravdess_audio"
    )
)

# Update CREMA-D paths → standardized audio
df_crema["path"] = df_crema["path"].apply(
    lambda p: p.replace(
        r"D:\SER_Cross\data\crema_d",
        r"D:\SER_Cross\data\processed\crema_d_audio"
    )
)


In [4]:
df_all = pd.concat([df_ravdess, df_crema], ignore_index=True)

print("Total samples:", len(df_all))
df_all.head()


Total samples: 7227


,path,emotion,speaker,dataset
0,D:\SER_Cross\data\processed\ravdess_audio\Acto...,neutral,actor_01,ravdess
1,D:\SER_Cross\data\processed\ravdess_audio\Acto...,neutral,actor_01,ravdess
2,D:\SER_Cross\data\processed\ravdess_audio\Acto...,neutral,actor_01,ravdess
3,D:\SER_Cross\data\processed\ravdess_audio\Acto...,neutral,actor_01,ravdess
4,D:\SER_Cross\data\processed\ravdess_audio\Acto...,neutral,actor_01,ravdess


In [5]:
print("Emotion distribution:")
print(df_all["emotion"].value_counts())

print("\nDataset distribution:")
print(df_all["dataset"].value_counts())


Emotion distribution:
emotion
happy      1463
sad        1463
angry      1463
fear       1463
neutral    1375
Name: count, dtype: int64

Dataset distribution:
dataset
crema_d    6171
ravdess    1056
Name: count, dtype: int64


In [6]:
speakers = df_all["speaker"].unique()
print("Total unique speakers:", len(speakers))


Total unique speakers: 115


In [7]:
from sklearn.model_selection import train_test_split

train_speakers, temp_speakers = train_test_split(
    speakers, test_size=0.30, random_state=42
)

val_speakers, test_speakers = train_test_split(
    temp_speakers, test_size=0.50, random_state=42
)


In [8]:
df_train = df_all[df_all["speaker"].isin(train_speakers)]
df_val   = df_all[df_all["speaker"].isin(val_speakers)]
df_test  = df_all[df_all["speaker"].isin(test_speakers)]

print("Train:", len(df_train))
print("Val:  ", len(df_val))
print("Test: ", len(df_test))


Train: 5068
Val:   1031
Test:  1128


In [9]:
assert set(df_train["speaker"]).isdisjoint(df_val["speaker"])
assert set(df_train["speaker"]).isdisjoint(df_test["speaker"])
assert set(df_val["speaker"]).isdisjoint(df_test["speaker"])

print("No speaker leakage ✔")


No speaker leakage ✔


In [10]:
import os

out_dir = r"D:\SER_Cross\data\processed"

df_train.to_csv(os.path.join(out_dir, "train.csv"), index=False)
df_val.to_csv(os.path.join(out_dir, "val.csv"), index=False)
df_test.to_csv(os.path.join(out_dir, "test.csv"), index=False)

print("Saved train.csv, val.csv, test.csv")


Saved train.csv, val.csv, test.csv


In [11]:
import os
print(os.listdir(r"D:\SER_Cross\data\processed"))


['crema_d_audio', 'crema_d_metadata.csv', 'ravdess_audio', 'ravdess_metadata.csv', 'test.csv', 'train.csv', 'val.csv']
